# DiffPO Confounding Diffusion Analysis

Compares DDIM denoising trajectories between DiffPO trained on clean IHDP (`naive_full`) and
DiffPO trained on confounded IHDP (`naive_conf`). See
`docs/superpowers/specs/2026-08-18-diffpo-confounding-diffusion-analysis-design.md`.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from scipy.stats import gaussian_kde

from src.config import Config
from src.data import load_ihdp, make_ihdp_confounded
from src.model import DiffPO, _DiffusionBase

/home/justin/msc_ai/individual-project/diffusion-irregular-ehr/.claude/worktrees/diffpo-confounding-diffusion-analysis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CLEAN_CHECKPOINT = "checkpoints/final_model_naive_full_2026-08-03T16_14_47.pth"
CONF_CHECKPOINT = "checkpoints/final_model_naive_conf_2026-08-05T10_28_18.pth"
N_PER_GROUP = 10  # subjects per momblack group
SEED = 0
DEVICE = torch.device("cpu")

In [3]:
with open("config/ihdp.yaml") as f:
    cfg = Config.model_validate(yaml.safe_load(f))

# Pin to the diffusion schedule the available naive_full/naive_conf checkpoints were actually
# trained with. config/ihdp.yaml's diffusion.num_steps/beta_end were bumped (100->200, 0.2->0.1)
# in a later "update best diffusion params" commit that post-dates every checkpoint currently on
# disk, so loading state_dict against the current config's schedule shapes fails
# (beta_sched/alpha_sched/alpha_bar_sched: checkpoint shape (100,) vs. config-derived shape
# (200,)). Overriding here, rather than editing config/ihdp.yaml itself, keeps this notebook
# self-documenting about which schedule these specific checkpoints require.
cfg.diffusion.num_steps = 100
cfg.diffusion.beta_end = 0.2

clean_model = DiffPO(cfg.model, cfg.diffusion)
clean_model.load_state_dict(torch.load(CLEAN_CHECKPOINT, map_location=DEVICE))
clean_model.eval()

conf_model = DiffPO(cfg.model, cfg.diffusion)
conf_model.load_state_dict(torch.load(CONF_CHECKPOINT, map_location=DEVICE))
conf_model.eval()

print("clean_model.L:", clean_model.L, "  conf_model.L:", conf_model.L)

clean_model.L: 100   conf_model.L: 100


In [4]:
train_ds_clean, val_ds_clean, test_ds_clean, y_std = load_ihdp(
    cfg.data.path,
    replication=cfg.data.replication,
    train_ratio=cfg.data.train_ratio,
    test_ratio=cfg.data.test_ratio,
)
train_ds_conf, val_ds_conf, test_ds_conf = (
    make_ihdp_confounded(ds, effect=cfg.data.confounder_effect)
    for ds in (train_ds_clean, val_ds_clean, test_ds_clean)
)


def _clip_value(train_ds):
    if not cfg.diffusion.clip_denoised:
        return None
    y_both = _DiffusionBase._assemble_yboth(train_ds.a, train_ds.y, train_ds.y_cf)
    return 2 * y_both.abs().max().item()


clip_value_clean = _clip_value(train_ds_clean)
clip_value_conf = _clip_value(train_ds_conf)
print("clip_value_clean:", clip_value_clean, "  clip_value_conf:", clip_value_conf)

clip_value_clean: 6.018764972686768   clip_value_conf: 7.191953659057617


In [5]:
rng = np.random.default_rng(SEED)
confounder = test_ds_clean.confounder
assert confounder is not None

idx_flipped = np.flatnonzero(confounder == 1)
idx_control = np.flatnonzero(confounder == 0)
assert len(idx_flipped) >= N_PER_GROUP, f"only {len(idx_flipped)} momblack==1 test subjects"
assert len(idx_control) >= N_PER_GROUP, f"only {len(idx_control)} momblack==0 test subjects"

subset_idx = np.concatenate(
    [
        rng.choice(idx_flipped, size=N_PER_GROUP, replace=False),
        rng.choice(idx_control, size=N_PER_GROUP, replace=False),
    ]
)
momblack_t = torch.as_tensor(confounder[subset_idx])
flipped_mask = momblack_t == 1
control_mask = momblack_t == 0

x = test_ds_clean.x[subset_idx]
a_clean = test_ds_clean.a[subset_idx]
a_conf = test_ds_conf.a[subset_idx]

# mu0/mu1 differ by condition: cfg.data.confounder_effect=0.4 (config/ihdp.yaml) means
# make_ihdp_confounded doesn't just relabel treatment, it shifts the *true* potential
# outcomes for momblack==1 subjects (Hill's mechanism, applied in raw-outcome units then
# renormalised -- see src/data.py). Each model must be scored against its own condition's
# ground truth, matching how experiment.py's evaluate() does it -- never mu0_clean for the
# confound model or vice versa.
mu0_clean = test_ds_clean.mu0[subset_idx]
mu1_clean = test_ds_clean.mu1[subset_idx]
mu0_conf = test_ds_conf.mu0[subset_idx]
mu1_conf = test_ds_conf.mu1[subset_idx]

# Data-setup claim from the spec: momblack==0 subjects must see identical (x, a) in both
# conditions (built-in control group); momblack==1 subjects must see exactly-flipped a.
assert torch.equal(a_clean[control_mask], a_conf[control_mask])
assert torch.equal(a_conf[flipped_mask], 1.0 - a_clean[flipped_mask])
assert torch.equal(test_ds_clean.x[subset_idx], test_ds_conf.x[subset_idx])
# mu0/mu1 must also agree on the control group (only momblack==1 subjects' true POs shift).
# allclose, not equal: make_ihdp_confounded round-trips mu0/mu1 through denorm()/renorm() even
# for momblack==0 subjects (where the shift itself is a mathematical no-op), and that float32
# round trip is not bit-exact -- observed diffs are ~1e-7, i.e. float32 machine-epsilon noise,
# not a sign the mechanism touched control subjects.
assert torch.allclose(mu0_clean[control_mask], mu0_conf[control_mask], atol=1e-6)
assert torch.allclose(mu1_clean[control_mask], mu1_conf[control_mask], atol=1e-6)

print(
    f"Subset: {len(subset_idx)} subjects "
    f"({int(flipped_mask.sum())} momblack==1, {int(control_mask.sum())} momblack==0)"
)

Subset: 20 subjects (10 momblack==1, 10 momblack==0)
